In [0]:
from pyspark.sql.window import Window
from pyspark.sql import functions as F


# 0. Create schema

spark.sql("CREATE SCHEMA IF NOT EXISTS hr_catalog_divansh.silver")


# 1. Read Bronze tables

emp = spark.table("hr_catalog_divansh.bronze.employees") \
    .withColumn("hire_date", F.to_date("hire_date", "yyyy-MM-dd")) \
    .withColumn("tenure_years",
        F.round(F.datediff(F.current_date(), F.col("hire_date")) / 365.25, 2)
    ).alias("emp")

sal = spark.table("hr_catalog_divansh.bronze.salaries") \
    .withColumn("effective_date", F.to_date("effective_date", "yyyy-MM-dd")) \
    .alias("sal")

dep = spark.table("hr_catalog_divansh.bronze.departments").alias("dep")


# 2. Latest salary (window)

salary_window = Window.partitionBy("employee_id").orderBy(F.desc("effective_date"))

latest_salary = (sal
    .withColumn("rn", F.row_number().over(salary_window))
    .filter(F.col("rn") == 1)
    .drop("rn")
    .withColumn(
        "total_monthly_compensation",
        F.col("base_salary") + F.col("base_salary") * F.col("bonus_pct")
    )
).alias("sal")


# 3. Join and enrich

enriched = (emp
    .join(latest_salary, "employee_id", "left")
    .join(dep, "department_id", "left")
    .select(
        "employee_id",
        F.col("emp.full_name"),
        F.col("emp.email"),
        "department_id",
        F.col("dep.department_name"),
        F.col("dep.location"),
        F.col("emp.job_title"),
        F.col("emp.hire_date"),
        F.col("emp.tenure_years"),
        F.col("emp.employment_type"),
        F.col("emp.manager_id"),
        F.col("sal.base_salary"),
        F.col("sal.bonus_pct"),
        F.col("sal.total_monthly_compensation"),
        F.col("sal.effective_date"),
        F.col("sal.currency"),
        F.col("emp._ingested_at").alias("_ingested_at")
    )
)


# 4. Write Silver table

enriched.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("hr_catalog_divansh.silver.employees_enriched")


# 5. Enable CDF (IMPORTANT)

spark.sql("""
ALTER TABLE hr_catalog_divansh.silver.employees_enriched
SET TBLPROPERTIES (delta.enableChangeDataFeed = true)
""")


# 6. Make a change (to generate CDF)

spark.sql("""
UPDATE hr_catalog_divansh.silver.employees_enriched
SET job_title = 'Senior Analyst'
WHERE employee_id IN (
  SELECT employee_id
  FROM hr_catalog_divansh.silver.employees_enriched
  LIMIT 1
)
""")


# 7. Check history (get version)

spark.sql("""
DESCRIBE HISTORY hr_catalog_divansh.silver.employees_enriched
""").show(truncate=False)


# 8. Read Change Data (SAFE)

spark.sql("""
SELECT *
FROM table_changes('hr_catalog_divansh.silver.employees_enriched', 1)
""").show(10)


# 9. Optimize

spark.sql("""
OPTIMIZE hr_catalog_divansh.silver.employees_enriched
ZORDER BY (department_id, hire_date)
""")